In [1]:
from ansys.aedt.core import Desktop, Hfss, Q3d
from pathlib import Path
import traceback
import pandas as pd


desktop = Desktop(version="2025.2", new_desktop=False, student_version=True, port=50051, close_on_exit=False)
q3d = Q3d(project="Project13", new_desktop=False, close_on_exit=False)


PyAEDT INFO: Python version 3.12.12 (main, Oct 31 2025, 23:00:21) [MSC v.1944 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.23.0.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\abhis\AppData\Local\Temp\pyaedt_abhis_2c671999-afd7-43d6-a945-ad8cf29b8b52.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Connecting to AEDT gRPC session on port 50051.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\ANSYS Student\v252\AnsysEM
PyAEDT INFO: Client application successfully started.
PyAEDT WARNING: Service Pack is not detected. PyAEDT is currently connecting in Insecure Mode.
PyAEDT WARNING: Please download and install latest Service Pack to use connect to AEDT in Secure Mode.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Python version 3.12.12 (main, Oct 31 2025, 23:00:21) [MSC v.1944 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.23.0.
PyAEDT INFO:

In [2]:
q3d.analyze(setup="Setup")


PyAEDT INFO: Project Project13 Saved correctly
PyAEDT INFO: Solving design setup Setup
PyAEDT INFO: Design setup Setup solved correctly in 0.0h 0.0m 5.0s


True

In [2]:
q3d.units.capacitance 

'pF'

In [ ]:
cap_data = q3d.post.get_solution_data(
    expressions=q3d.post.get_all_report_quantities()["Matrix"]["Setup : AdaptivePass"][
        "C Matrix"
    ],
    context="Original",
    report_category="Matrix",
    setup_sweep_name="Setup : AdaptivePass",
    variations={"Pass": 8},
)
cap_mat_real = cap_data.full_matrix_real_imag[0]
pass_c_df = (
    pd.DataFrame(
        [(*k[2:-1].split(","), v[0, -1]) for k, v in cap_mat_real.items()],
        columns=["node1", "node2", "cap"],
    )
    .pivot(index="node1", columns="node2", values="cap")
    .rename_axis(None, axis=1)
    .rename_axis(None, axis=0)
)
pass_c_df * 1e3

PyAEDT INFO: Parsing C:\Users\abhis\OneDrive\Documents\Ansoft\Project13.aedt.
PyAEDT INFO: File C:\Users\abhis\OneDrive\Documents\Ansoft\Project13.aedt correctly loaded. Elapsed time: 0m 0sec
PyAEDT INFO: aedt file load time 0.042510032653808594


PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Post class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 4sec
PyAEDT INFO: Solution Data Correctly Loaded.
Time to initialize solution data:0.007309913635253906
Time to initialize solution data:0.09796857833862305


,bus1_connector_pad_Q1,bus2_connector_pad_Q1,ground_main_plane,pad_bot_Q1,pad_top_Q1,readout_connector_pad_Q1
bus1_connector_pad_Q1,50.665811,-0.422541,-34.047248,-1.566166,-13.500399,-0.204514
bus2_connector_pad_Q1,-0.422541,54.945543,-36.349870,-14.289852,-1.837534,-1.028348
ground_main_plane,-34.047248,-36.349870,239.244876,-31.550365,-38.192018,-37.189727
pad_bot_Q1,-1.566166,-14.289852,-31.550365,99.558389,-30.711097,-19.241255
pad_top_Q1,-13.500399,-1.837534,-38.192018,-30.711097,89.198763,-2.225892
readout_connector_pad_Q1,-0.204514,-1.028348,-37.189727,-19.241255,-2.225892,61.006694


In [37]:
last_cap_qty_name=  f"cap ({q3d.units.capacitance})"
last_cap_data = q3d.post.get_solution_data(
    expressions=q3d.post.get_all_report_quantities()["Matrix"]["Setup : LastAdaptive"]["C Matrix"],
    context="Original",
    setup_sweep_name="Setup : LastAdaptive",
    variations={"Pass": ["1"]})
last_pass_c_df = pd.DataFrame([(*k[2:-1].split(','),  v[0, -1])for k, v in last_cap_data.full_matrix_mag_phase[0].items()], columns=["node1", "node2", last_cap_qty_name   ]).pivot(index="node1", columns="node2", values=last_cap_qty_name).rename_axis(None, axis=1).rename_axis(None, axis=0)
last_pass_c_df

PyAEDT WARNING: No report category provided. Automatically identified Matrix
PyAEDT INFO: Solution Data Correctly Loaded.
Time to initialize solution data:0.0073757171630859375
Time to initialize solution data:0.09889340400695801


,bus1_connector_pad_Q1,bus2_connector_pad_Q1,ground_main_plane,pad_bot_Q1,pad_top_Q1,readout_connector_pad_Q1
bus1_connector_pad_Q1,0.050666,0.000423,0.034047,0.001566,0.013500,0.000205
bus2_connector_pad_Q1,0.000423,0.054946,0.036350,0.014290,0.001838,0.001028
ground_main_plane,0.034047,0.036350,0.239245,0.031550,0.038192,0.037190
pad_bot_Q1,0.001566,0.014290,0.031550,0.099558,0.030711,0.019241
pad_top_Q1,0.013500,0.001838,0.038192,0.030711,0.089199,0.002226
readout_connector_pad_Q1,0.000205,0.001028,0.037190,0.019241,0.002226,0.061007


In [20]:
q3d.cleanup_solution()

True

In [10]:
desktop.release_desktop(close_projects=False, close_on_exit=False)


True